> **LLM note:** The following cell was generated with Claude assistance (Level 4 substantial).
> Drafted by Claude, for making optuna plots

In [ ]:
import os, matplotlib.pyplot as plt
import optuna
import sys, io
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
)

os.makedirs('PLOTS', exist_ok=True)


# Tee print output to both stdout and a buffer
class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, s):  [st.write(s) for st in self.streams]
    def flush(self):     [st.flush() for st in self.streams]

_buf = io.StringIO()
sys.stdout = _Tee(sys.__stdout__, _buf)

# ── Aesthetic helpers (matches M1/M2/M3 plots ───────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'font.weight':       'light',
    'axes.titleweight':  'normal',
    'axes.labelweight':  'normal',
    'axes.titlesize':    10,
    'axes.labelsize':    9,
    'legend.fontsize':   8,
    'legend.frameon':    False,
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
})

def style_axis(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_linewidth(0.5)
        ax.spines[s].set_color('#666')
    ax.tick_params(width=0.5, length=3)
    ax.grid(alpha=0.25, lw=0.5)

# ── Study registry ───────────────────────────────────────────────────
STUDIES = [
    # (label,        db_filename,                              study_name)
    ('LSTM',         'optuna_forecast_LSTM_bsm2.db',         'forecast_LSTM_bsm2'),
    ('GRU',          'optuna_forecast_GRU_bsm2.db',          'forecast_GRU_bsm2'),
    ('Transformer',  'optuna_forecast_Transformer_bsm2.db',  'forecast_Transformer_bsm2'),
    ('VAE',          'optuna__anamoly_vae_bsm2.db',          'vae_bsm2'),
]

# ── Load every study, print summary ──────────────────────────────────
studies = {}
print(f'\n{"="*72}')
print(f'  Optuna study summary')
print(f'{"="*72}')

for label, db, name in STUDIES:
    if not os.path.exists(db):
        print(f'\n[{label}] SKIPPED — database file not found: {db}')
        continue
    storage = f'sqlite:///{db}'
    try:
        study = optuna.load_study(study_name=name, storage=storage)
    except Exception as e:
        print(f'\n[{label}] ERROR loading study: {e}')
        continue
    studies[label] = study

    n_complete = sum(1 for t in study.trials
                     if t.state == optuna.trial.TrialState.COMPLETE)
    n_pruned   = sum(1 for t in study.trials
                     if t.state == optuna.trial.TrialState.PRUNED)
    n_failed   = sum(1 for t in study.trials
                     if t.state == optuna.trial.TrialState.FAIL)

    print(f'\n[{label}]')
    print(f'  Trials: {len(study.trials)} total  '
          f'({n_complete} complete, {n_pruned} pruned, {n_failed} failed)')
    print(f'  Best value:   {study.best_value:.6f}')
    print(f'  Best params:')
    for k, v in study.best_params.items():
        if isinstance(v, float):
            print(f'    {k:<15s} = {v:.6g}')
        else:
            print(f'    {k:<15s} = {v}')

print(f'\n{"="*72}\n')

# ─────────────────────────────────────────────────────────────────────
# PLOTS — Optuna's matplotlib backend ships with its own default styling
# (grey background, large fonts, default title). We strip those after
# the plot is generated and apply our own consistent aesthetic.
# ─────────────────────────────────────────────────────────────────────
def restyle_optuna_ax(ax, new_title):
    """Strip Optuna's default styling and apply our project aesthetic."""
    fig = ax.figure
    fig.set_size_inches(7.5, 4.5)
    # Optuna sets titles with loc='left' (e.g. "Hyperparameter Importances"),
    # which our ax.get_title() doesn't see by default. Clear all three locs.
    for loc in ('left', 'center', 'right'):
        ax.set_title('', loc=loc)
    # Clear any figure-level suptitle too
    if fig._suptitle is not None:
        fig._suptitle.set_visible(False)
    # Apply our title
    ax.set_title(new_title, loc='center', pad=8, fontsize=10)
    # Optuna calls plt.style.use("ggplot") which pollutes globals (grey bg,
    # thick spines). Reset to the rcParams we set at module-level.
    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')
    # Restyle axis labels in our font sizes
    ax.xaxis.label.set_size(9)
    ax.yaxis.label.set_size(9)
    ax.tick_params(labelsize=8)
    # Spines and grid match the rest of the project
    style_axis(ax)
    # Legend
    leg = ax.get_legend()
    if leg is not None:
        leg.set_frame_on(False)
        for txt in leg.get_texts():
            txt.set_fontsize(8)
    fig.tight_layout()
    return fig

for label, study in studies.items():
    n_complete = sum(1 for t in study.trials
                     if t.state == optuna.trial.TrialState.COMPLETE)

    # ── Optimisation history ─────────────────────────────────────────
    ax = plot_optimization_history(study)
    fig = restyle_optuna_ax(ax, f'Optimisation history — {label}')
    out = f'PLOTS/optuna_history_{label}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved → {out}')

    # ── Parameter importance (needs enough completed trials) ────────
    if n_complete >= 5:
        try:
            ax = plot_param_importances(study)
            fig = restyle_optuna_ax(ax, f'Hyperparameter importance — {label}')
            out = f'PLOTS/optuna_importance_{label}.png'
            fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
            plt.close(fig)
            print(f'Saved → {out}')
        except Exception as e:
            print(f'  [{label}] importance plot skipped: {e}')
    else:
        print(f'  [{label}] importance plot skipped: too few completed trials')

print(f'\n{"="*72}')
print(f'  Done. Plots saved to PLOTS/')
print(f'{"="*72}')

# Restore stdout and save captured output
sys.stdout = sys.__stdout__
with open('PLOTS/optuna_summary.txt', 'w', encoding='utf-8') as f:
    f.write(_buf.getvalue())
print(f'Saved → PLOTS/optuna_summary.txt')

C:\Users\jekan6875\AppData\Local\Temp\ipykernel_17316\2639219649.py:133: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_optimization_history(study)
C:\Users\jekan6875\AppData\Local\Temp\ipykernel_17316\2639219649.py:143: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_param_importances(study)
C:\Users\jekan6875\AppData\Local\Temp\ipykernel_17316\2639219649.py:133: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_optimization_history(study)
C:\Users\jekan6875\AppData\Local\Temp\ipykernel_17316\2639219649.py:143: ExperimentalWarning: optuna.visualization.matplotli